In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install wandb

In [3]:
import wandb
from kaggle_secrets import UserSecretsClient

api_key = UserSecretsClient().get_secret("WAND")
wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
import random, re

import torch
import torch.nn as nn

from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup

In [5]:
random.seed(33)
np.random.seed(33)
torch.manual_seed(33)

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(33)

In [6]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [8]:
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [9]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      500 non-null    int64 
 1   prompt  500 non-null    object
 2   A       500 non-null    object
 3   B       500 non-null    object
 4   C       500 non-null    object
 5   D       500 non-null    object
 6   E       500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


In [10]:
OPTIONS = ["A", "B", "C", "D", "E"]

def split_prompt(prompt):
    text = str(prompt).strip()
    prefix, suffix = '', ''

    ci = text.find(': ')
    if ci != -1:
        prefix = text[:ci].strip()
        text = text[ci + 1:].strip()

    n_q = text.count('?')

    if n_q == 1:
        qi = text.index('?')
        suffix = text[qi + 1:].strip()
        text = text[:qi + 1].strip()

    elif n_q == 2:
        first_q = text.index('?')
        prefix = text[:first_q + 1].strip()

        text = text[first_q + 1:].strip()
        second_q = text.index('?')

        suffix = text[second_q + 1:].strip()
        text = text[:second_q + 1].strip()

    return (prefix or None), text, (suffix or None)


def clean(text, is_prompt=True):
    if(is_prompt):
        text = split_prompt(text)[1]
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# def tokenize(text, is_prompt=True):
#     return clean(text, is_prompt).split()


def ap_at_3(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0


def map_at_3(actuals, predictions):
    return float(np.mean([ap_at_3(a, p) for a, p in zip(actuals, predictions)]))


def make_submission(ids, predictions):
    return pd.DataFrame({
        'id': ids,
        'prediction': [' '.join(p[:3]) for p in predictions]
    })

def probabilities_to_predictions(probabilities):
    return [
        [OPTIONS[idx] for idx in np.argsort(-probability_row)][:3]
        for probability_row in probabilities
    ]

In [11]:
df['q_clean'] = df['prompt'].apply(clean)
test_df['q_clean'] = test_df['prompt'].apply(clean)

for opt in OPTIONS:
    df[f'{opt}_clean'] = df[opt].apply(lambda text: clean(text, is_prompt=False))
    test_df[f'{opt}_clean'] = test_df[opt].apply(lambda text: clean(text, is_prompt=False))

print(df.info())
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       2000 non-null   int64 
 1   prompt   2000 non-null   object
 2   A        2000 non-null   object
 3   B        2000 non-null   object
 4   C        2000 non-null   object
 5   D        2000 non-null   object
 6   E        2000 non-null   object
 7   answer   2000 non-null   object
 8   q_clean  2000 non-null   object
 9   A_clean  2000 non-null   object
 10  B_clean  2000 non-null   object
 11  C_clean  2000 non-null   object
 12  D_clean  2000 non-null   object
 13  E_clean  2000 non-null   object
dtypes: int64(1), object(13)
memory usage: 218.9+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       500 non-null    int64 
 1   prompt   500 non-null 

In [12]:
print("Empty questions:", (df['q_clean'].str.len() == 0).sum())

for opt in OPTIONS:
    empty_count = (df[f'{opt}_clean'].str.len() == 0).sum()
    print(opt, empty_count)

Empty questions: 0
A 0
B 0
C 0
D 0
E 0


In [13]:
unique_qs = df['q_clean'].unique()
(len(df), len(unique_qs), len(set(unique_qs)))

(2000, 252, 252)

In [14]:
NUM_FOLDS = 5

rng = np.random.default_rng(33)
shuffled_qs = unique_qs.copy()
rng.shuffle(shuffled_qs)

fold_qs = np.array_split(shuffled_qs, NUM_FOLDS)

for fold in range(NUM_FOLDS):
    print(f"Fold {fold}: {len(fold_qs[fold])} validation questions")

Fold 0: 51 validation questions
Fold 1: 51 validation questions
Fold 2: 50 validation questions
Fold 3: 50 validation questions
Fold 4: 50 validation questions


In [15]:
def create_pair_dataframe(source_df, has_labels=True):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        for opt in OPTIONS:
            pair_row = {
                'id': row.id,
                'option': opt,
                'question': row.q_clean,
                'option_text': getattr(row, f'{opt}_clean'),
                'tokens': getattr(row, f'{opt}_clean')
            }
            if has_labels:
                pair_row['label'] = int(row.answer == opt)
            
            pair_rows.append(pair_row)

    return pd.DataFrame(pair_rows)

In [16]:
train_pairs = create_pair_dataframe(df, has_labels=True)
test_pairs = create_pair_dataframe(test_df, has_labels=False)

print("Train pairs:", train_pairs.shape)
print("Test pairs:", test_pairs.shape)

Train pairs: (10000, 6)
Test pairs: (2500, 5)


In [17]:
train_pairs.head()

,id,option,question,option_text,tokens,label
0,1,A,what is martin heidegger s view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans exist wi...,0
1,1,B,what is martin heidegger s view on the relatio...,martin heidegger believes that humans do not e...,martin heidegger believes that humans do not e...,1
2,1,C,what is martin heidegger s view on the relatio...,martin heidegger does not believe in the exist...,martin heidegger does not believe in the exist...,0
3,1,D,what is martin heidegger s view on the relatio...,martin heidegger believes that the relationshi...,martin heidegger believes that the relationshi...,0
4,1,E,what is martin heidegger s view on the relatio...,martin heidegger believes that time is an illu...,martin heidegger believes that time is an illu...,0


In [18]:
MINILM_CONFIG = {
    'name': 'minilm',
    'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
    'max_length': 128,
    'batch_size': 32,
    'learning_rate': 5e-5,
    'epochs': 15,
    'patience': 5
}

DEBERTA_CONFIG = {
    'name': 'deberta',
    'model_name': 'microsoft/deberta-v3-small',
    'max_length': 128,
    'batch_size': 8,
    'learning_rate': 2e-5,
    'epochs': 10,
    'patience': 4
}

In [ ]:
tokenizers = {
    MINILM_CONFIG['name']: AutoTokenizer.from_pretrained(
        MINILM_CONFIG['model_name']
    ),
    DEBERTA_CONFIG['name']: AutoTokenizer.from_pretrained(
        DEBERTA_CONFIG['model_name']
    )
}

In [20]:
def encode_groups(pair_df, tokenizer, max_len, has_labels=True):
    samples = []

    for question_id, group in pair_df.groupby('id', sort=False):
        group = group.sort_values('option')

        encoded = tokenizer(
            group['question'].tolist(),
            group['option_text'].tolist(),
            truncation=True,
            max_length=max_len,
            padding='max_length',
            return_tensors='pt'
        )

        sample = {
            'input_ids': encoded['input_ids'],
            'attention_mask': encoded['attention_mask'],
            'id': question_id
        }

        if has_labels:
            label_idx = group['label'].tolist().index(1)
            sample['label'] = torch.tensor(label_idx, dtype=torch.long)

        samples.append(sample)

    return samples

In [21]:
def create_model(model_name):
    model = AutoModelForMultipleChoice.from_pretrained(
        model_name,
        dtype=torch.float32
    )
    return model

In [22]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, total_samples = 0.0, 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

In [23]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    ids_list, probabilities_list = [], []
    predictions, actual_answers = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        ids_list.extend(batch['id'].cpu().numpy().tolist())
        probabilities_list.extend(probabilities)

        labels_cpu = labels.cpu().numpy()

        for probability_row, label in zip(probabilities, labels_cpu):
            ranked_options = [OPTIONS[idx] for idx in np.argsort(-probability_row)]
            predictions.append(ranked_options[:3])
            actual_answers.append(OPTIONS[label])

    return {
        'ids': ids_list,
        'loss': total_loss / total_samples,
        'map3': map_at_3(actual_answers, predictions),
        'probabilities': np.array(probabilities_list),
        'predictions': predictions,
        'actual_answers': actual_answers
    }

In [24]:
@torch.no_grad()
def predict_test(model, loader, device):
    model.eval()

    ids_list, probabilities_list = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model(input_ids, attention_mask).logits
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

        ids_cpu = batch['id'].cpu().numpy()

        ids_list.extend(ids_cpu.tolist())
        probabilities_list.extend(probabilities)

    return ids_list, np.array(probabilities_list)

In [25]:
def run_fold(fold, fold_qs, config, run):
    model_name = config['model_name']
    experiment_name = config['name']
    tokenizer = tokenizers[experiment_name]

    val_qs = set(fold_qs[fold])
    train_qs = set(unique_qs) - val_qs

    fold_train_pairs = train_pairs[train_pairs['question'].isin(train_qs)].reset_index(drop=True)
    fold_val_pairs = train_pairs[train_pairs['question'].isin(val_qs)].reset_index(drop=True)

    train_samples = encode_groups(fold_train_pairs, tokenizer, config['max_length'], has_labels=True)
    val_samples = encode_groups(fold_val_pairs, tokenizer, config['max_length'], has_labels=True)
    test_samples = encode_groups(test_pairs, tokenizer, config['max_length'], has_labels=False)

    train_loader = DataLoader(train_samples, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_samples, batch_size=config['batch_size']*2, shuffle=False)
    test_loader = DataLoader(test_samples, batch_size=config['batch_size'] * 2, shuffle=False)

    model = create_model(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=0.01)

    total_steps = len(train_loader) * config['epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_map3 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    history = []
    checkpoint_path = f'best_{experiment_name}_fold{fold}.pt'

    for epoch in range(1, config['epochs'] + 1):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        validation_result = evaluate(model, val_loader, device)

        val_loss = validation_result['loss']
        val_map3 = validation_result['map3']

        history.append({
            'model': experiment_name,
            'fold': fold,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_map3': val_map3
        })

        print(f'{experiment_name} | Fold: {fold} | Epoch: {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val MAP@3: {val_map3:.4f}')

        run.log({
            'fold': fold,
            'epoch': epoch,
            f'{experiment_name}/loss': train_loss,
            f'{experiment_name}/loss': val_loss,
            f'{experiment_name}/map3': val_map3
        })

        if val_map3 > best_map3:
            best_map3 = val_map3
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(model.state_dict(), checkpoint_path)
            print(f'{experiment_name} | Fold: {fold} | Best model saved: MAP@3={best_map3:.4f}')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= config['patience']:
                print(f'{experiment_name} | Fold: {fold} | Early stopping after epoch {epoch}.')
                break

    print(f'\n{experiment_name} | Fold {fold} training complete')
    print(f'{experiment_name} | Fold {fold}: Best epoch: {best_epoch}')
    print(f'{experiment_name} | Fold {fold}: Best validation MAP@3: {best_map3}')


    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()

    best_validation_result = evaluate(model, val_loader, device)
    test_ids, test_probabilities = predict_test(model, test_loader, device)

    run.summary[f'{experiment_name}/fold_{fold}_best_epoch'] = best_epoch
    run.summary[f'{experiment_name}/fold_{fold}_best_map3'] = best_map3

    result = {
        'model': experiment_name,
        'fold': fold,
        'history': pd.DataFrame(history),
        'best_epoch': best_epoch,
        'best_map3': best_map3,
        'oof_ids': best_validation_result['ids'],
        'oof_probabilities': best_validation_result['probabilities'],
        'oof_actuals': best_validation_result['actual_answers'],
        'test_ids': test_ids,
        'test_probabilities': test_probabilities
    }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [26]:
run = wandb.init(
    project='23f2003391-t22026',
    name='minilm-deberta-oof-ensemble',
    config={
        'minilm_model': MINILM_CONFIG['model_name'],
        'minilm_epochs': MINILM_CONFIG['epochs'],
        'minilm_batch_size': MINILM_CONFIG['batch_size'],
        'minilm_learning_rate': MINILM_CONFIG['learning_rate'],
        'deberta_model': DEBERTA_CONFIG['model_name'],
        'deberta_epochs': DEBERTA_CONFIG['epochs'],
        'deberta_batch_size': DEBERTA_CONFIG['batch_size'],
        'deberta_learning_rate': DEBERTA_CONFIG['learning_rate'],
        'folds': NUM_FOLDS
    }
)

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260731_121731-595ec4as
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run minilm-deberta-oof-ensemble
wandb: ⭐️ View project at https://wandb.ai/23f2003391-dl-genai-project/23f2003391-t22026
wandb: 🚀 View run at https://wandb.ai/23f2003391-dl-genai-project/23f2003391-t22026/runs/595ec4as


In [ ]:
minilm_oof_by_id = {}
deberta_oof_by_id = {}

minilm_test_fold_probabilities = []
deberta_test_fold_probabilities = []

fold_results = []

for fold in range(NUM_FOLDS):
    print(f'FOLD: {fold}')

    minilm_result = run_fold(fold, fold_qs, MINILM_CONFIG, run)

    for question_id, probability_row in zip(minilm_result['oof_ids'], minilm_result['oof_probabilities']):
        minilm_oof_by_id[int(question_id)] = probability_row

    minilm_test_ids = minilm_result['test_ids']
    minilm_test_fold_probabilities.append(minilm_result['test_probabilities'])

    deberta_result = run_fold(fold, fold_qs, DEBERTA_CONFIG, run)

    for question_id, probability_row in zip(deberta_result['oof_ids'], deberta_result['oof_probabilities']):
        deberta_oof_by_id[int(question_id)] = probability_row

    deberta_test_ids = deberta_result['test_ids']
    deberta_test_fold_probabilities.append(deberta_result['test_probabilities'])

    fold_results.extend([minilm_result, deberta_result])

In [28]:
history_df = pd.concat(
    [result['history'].assign(fold=result['fold']) for result in fold_results],
    ignore_index=True
)
history_df

,model,fold,epoch,train_loss,val_loss,val_map3
0,minilm,0,1,1.606250,1.605289,0.469897
1,minilm,0,2,1.215329,1.648102,0.495595
2,minilm,0,3,0.360185,1.991650,0.506975
3,minilm,0,4,0.181365,1.767362,0.521659
4,minilm,0,5,0.086479,1.956501,0.512482
...,...,...,...,...,...,...
70,deberta,4,3,0.300657,1.639960,0.674200
71,deberta,4,4,0.155325,1.422320,0.636535
72,deberta,4,5,0.084117,1.964767,0.650188
73,deberta,4,6,0.042386,1.658576,0.614407


In [29]:
train_ids = df['id'].astype(int).tolist()
actual_answers = df['answer'].tolist()

minilm_oof_probabilities = np.array([
    minilm_oof_by_id[question_id]
    for question_id in train_ids
])

deberta_oof_probabilities = np.array([
    deberta_oof_by_id[question_id]
    for question_id in train_ids
])

minilm_oof_predictions = probabilities_to_predictions(minilm_oof_probabilities)
deberta_oof_predictions = probabilities_to_predictions(deberta_oof_probabilities)

minilm_oof_map3 = map_at_3(actual_answers, minilm_oof_predictions)
deberta_oof_map3 = map_at_3(actual_answers, deberta_oof_predictions)

print("MiniLM OOF MAP@3:", minilm_oof_map3)
print("DeBERTa OOF MAP@3:", deberta_oof_map3)

run.summary['oof/minilm_map3'] = minilm_oof_map3
run.summary['oof/deberta_map3'] = deberta_oof_map3

MiniLM OOF MAP@3: 0.5614166666666667
DeBERTa OOF MAP@3: 0.6309166666666668


In [30]:
blend_results = []

for minilm_weight in np.arange(0.0, 1.01, 0.05):
    deberta_weight = 1.0 - minilm_weight

    blended_oof_probabilities = (minilm_weight * minilm_oof_probabilities + deberta_weight * deberta_oof_probabilities)

    blended_oof_predictions = probabilities_to_predictions(blended_oof_probabilities)
    blended_oof_map3 = map_at_3(actual_answers, blended_oof_predictions)

    blend_results.append({
        'minilm_weight': float(minilm_weight),
        'deberta_weight': float(deberta_weight),
        'map3': blended_oof_map3
    })

blend_results_df = pd.DataFrame(blend_results)
blend_results_df

,minilm_weight,deberta_weight,map3
0,0.00,1.00,0.630917
1,0.05,0.95,0.631667
2,0.10,0.90,0.630667
3,0.15,0.85,0.636250
4,0.20,0.80,0.638667
5,0.25,0.75,0.645583
6,0.30,0.70,0.646417
7,0.35,0.65,0.643500
8,0.40,0.60,0.631333
9,0.45,0.55,0.625083


In [31]:
best_blend = max(blend_results, key=lambda result: result['map3'])

best_minilm_weight = best_blend['minilm_weight']
best_deberta_weight = best_blend['deberta_weight']
best_blended_oof_map3 = best_blend['map3']

print("Best MiniLM weight:", best_minilm_weight)
print("Best DeBERTa weight:", best_deberta_weight)
print("Best blended OOF MAP@3:", best_blended_oof_map3)

run.summary['ensemble/minilm_weight'] = best_minilm_weight
run.summary['ensemble/deberta_weight'] = best_deberta_weight
run.summary['ensemble/oof_map3'] = best_blended_oof_map3

Best MiniLM weight: 0.30000000000000004
Best DeBERTa weight: 0.7
Best blended OOF MAP@3: 0.6464166666666668


In [32]:
run.finish()

wandb: updating run metadata
wandb: uploading summary, console lines 317-321
wandb: 
wandb: Run history:
wandb: deberta/loss ▂▁▂▃▃▃▂▂▂▅▆█▇▇▁▃▃▄▅▁▁▃▃██▁▁▃▁▄▃▅
wandb: deberta/map3 ▁▅▄▃▃▃▄▅▄▅▅▅▅▅▇▃▅▅▄▅█▄▄▄▁▆▇█▇▇▆▆
wandb:        epoch ▂▃▃▄▅█▂▃▃▄▂▃▄▁▂▄▅▅▁▂▃▄▅▂▃▂▃▄▅▅▃▃▄▁▂▄▅▂▃▅
wandb:         fold ▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆███████
wandb:  minilm/loss ▂▃▅▃▄▄▅▆▅▅▅▅▅▁▂▃▃▄▄▂▃▄▄▅▅▆▂▂▃▄▄▆▆▇▂▁▄▅▄█
wandb:  minilm/map3 ▃▄▅▅▅▅▆▇▆▆▆▆▅▆▅▅▅▅▃▁▆█▇▇▇▆▂▃▇▆▆▅▅▄▁▇▃▄▃▅
wandb: 
wandb: Run summary:
wandb: deberta/fold_0_best_epoch 2
wandb:  deberta/fold_0_best_map3 0.59912
wandb: deberta/fold_1_best_epoch 4
wandb:  deberta/fold_1_best_map3 0.59641
wandb: deberta/fold_2_best_epoch 1
wandb:  deberta/fold_2_best_map3 0.63569
wandb: deberta/fold_3_best_epoch 2
wandb:  deberta/fold_3_best_map3 0.66202
wandb: deberta/fold_4_best_epoch 3
wandb:  deberta/fold_4_best_map3 0.6742
wandb:                       +21 ...
wandb: 
wandb: 🚀 View run minilm-deberta-oof-ensemble at: https://wandb.ai/23f2003391-dl-genai-projec

In [33]:
minilm_test_probabilities = np.mean(np.stack(minilm_test_fold_probabilities), axis=0)
deberta_test_probabilities = np.mean(np.stack(deberta_test_fold_probabilities), axis=0)

final_test_probabilities = (best_minilm_weight * minilm_test_probabilities + best_deberta_weight * deberta_test_probabilities)
final_test_predictions = probabilities_to_predictions(final_test_probabilities)

In [34]:
submission = make_submission(minilm_test_ids, final_test_predictions)

submission.to_csv('submission.csv', index=False)
submission.head(5)

,id,prediction
0,1,A C B
1,2,B E A
2,3,B E D
3,4,E C B
4,5,C A E
